In [1]:
import requests
import torch
import time
import psutil
import subprocess

import pandas as pd

from datasets import load_dataset

In [2]:
ds = load_dataset("cornell-movie-review-data/rotten_tomatoes")

test = ds['test'].to_pandas()

test.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1066 entries, 0 to 1065
Data columns (total 2 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   text    1066 non-null   object
 1   label   1066 non-null   int64 
dtypes: int64(1), object(1)
memory usage: 16.8+ KB


In [3]:
test['label'] = test['label'].apply(lambda x: 'positive' if x == 1 else 'negative')

labels = test['label'].unique()

test

,text,label
0,lovingly photographed in the manner of a golde...,positive
1,consistently clever and suspenseful .,positive
2,"it's like a "" big chill "" reunion of the baade...",positive
3,the story gives ample opportunity for large-sc...,positive
4,"red dragon "" never cuts corners .",positive
...,...,...
1061,a terrible movie that some people will neverth...,negative
1062,there are many definitions of 'time waster' bu...,negative
1063,"as it stands , crocodile hunter has the hurrie...",negative
1064,the thing looks like a made-for-home-video qui...,negative


In [4]:
def get_ollama_memory_usage(port=11434):
    """
    Finds the process listening on the given port using psutil
    and returns its memory usage in bytes (RSS).
    Returns None if the process isn't found or can't be accessed.
    """
    for proc in psutil.process_iter(['pid', 'name']):
        try:
            # Call proc.connections() to see if it's listening on the desired port
            for conn in proc.connections(kind='inet'):
                if conn.laddr.port == port:
                    # Found the process that listens on port=11434
                    memory_info = proc.memory_info()
                    return memory_info.rss  # in bytes
        except (psutil.AccessDenied, psutil.NoSuchProcess):
            pass
    
    # If no process was found
    return None

get_ollama_memory_usage()

C:\Users\Rafael\AppData\Local\Temp\ipykernel_21620\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


48398336

In [5]:
def get_gpu_memory_usage():
    """
    Returns a list of used memory (in MB) for each GPU.
    """
    # Use nvidia-smi with the --query-gpu and --format flags to get just the memory usage
    command = [
        "nvidia-smi",
        "--query-gpu=memory.used",  # You can also add memory.free, name, etc.
        "--format=csv,noheader,nounits"  # CSV output with no header or units
    ]
    try:
        output = subprocess.check_output(command)
        # Decode the output from bytes to string
        output_str = output.decode("utf-8").strip()
        # Each line corresponds to one GPU's memory usage
        usage_values = [int(x) for x in output_str.split("\n")]
        return usage_values[0]
    except subprocess.CalledProcessError as e:
        print("Error running nvidia-smi:", e)
        return []


In [6]:
def classify(text, labels):

    url = "http://localhost:11434/api/chat"

    payload = {
        "model": "gemma3",
        "messages" : [
            {"role": "system", "content": "You are a classification assistant. Your objective is to read the provided text and classify it according to the task and labels described. You are capable of handling binary classification tasks based on user instructions."},
            {"role": "user", "content": f"Classify the following text based on the task: Sentiment analysis of movie reviews. Only respond with the label that best describe the text. The possible labels are: {', '.join(labels)}. Text: {text}"}
        ],
        "stream": False,
        "options": {
            "temperature": 0
        }
    }

    start_time = time.time()
    response = requests.post(url, json=payload)
    response_time = time.time() - start_time

    vram_usage = get_gpu_memory_usage()

    ram_usage_bytes = get_ollama_memory_usage(port=11434) / (1024 * 1024)

    response = response.json()
    total_time = response['total_duration'] / 1_000_000_000
    content = response['message']['content'].lower()

    if 'positive' in content:
        content = 'positive'
    elif 'negative' in content:
        content = 'negative'
    else:
        content = 'error'

    return content, response_time, vram_usage, ram_usage_bytes, total_time

In [7]:
# apply the classify function to the test set. create one column for each output
test[['prediction', 'response_time', 'vram_usage', 'ram_usage', 'total_time']] = test['text'].apply(lambda x: classify(x, labels)).apply(pd.Series)

C:\Users\Rafael\AppData\Local\Temp\ipykernel_21620\1313842216.py:10: DeprecationWarning: connections() is deprecated and will be removed; use net_connections() instead
  for conn in proc.connections(kind='inet'):


In [8]:
test.to_csv('results/gemma_ZS_binary2.csv', index=False)
test

,text,label,prediction,response_time,vram_usage,ram_usage,total_time
0,lovingly photographed in the manner of a golde...,positive,positive,4.319057,5397,119.800781,2.276051
1,consistently clever and suspenseful .,positive,positive,2.141353,5405,123.515625,0.094539
2,"it's like a "" big chill "" reunion of the baade...",positive,positive,2.296317,5394,126.609375,0.256442
3,the story gives ample opportunity for large-sc...,positive,positive,2.221831,5389,126.753906,0.182578
4,"red dragon "" never cuts corners .",positive,negative,2.375242,5381,126.843750,0.325945
...,...,...,...,...,...,...,...
1061,a terrible movie that some people will neverth...,negative,negative,2.294686,5176,102.066406,0.260150
1062,there are many definitions of 'time waster' bu...,negative,negative,2.324098,5176,102.066406,0.278247
1063,"as it stands , crocodile hunter has the hurrie...",negative,negative,2.385376,5176,102.066406,0.337858
1064,the thing looks like a made-for-home-video qui...,negative,negative,2.233450,5176,102.066406,0.204791


In [9]:
y_pred = test['prediction']
y_true = test['label']

#import acc, f1_score, precision and recall from sklearn
from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score

#calculate the accuracy of the model
accuracy = accuracy_score(y_true, y_pred)
print('Accuracy: %f' % accuracy)
f1 = f1_score(y_true, y_pred, average='weighted')
print('F1 score: %f' % f1)
precision = precision_score(y_true, y_pred, average='weighted')
print('Precision: %f' % precision)
recall = recall_score(y_true, y_pred, average='weighted')
print('Recall: %f' % recall)

Accuracy: 0.850844
F1 score: 0.849328
Precision: 0.869855
Recall: 0.850844


c:\Users\Rafael\Desktop\Prog\TCC\venv\Lib\site-packages\sklearn\metrics\_classification.py:1531: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


In [10]:
# get average response time, vram usage and ram usage
response_time_avg = test['response_time'].mean()
vram_usage_avg = test['vram_usage'].mean()
ram_usage_avg = test['ram_usage'].mean()
total_time_avg = test['total_time'].mean()

print(f'Average response time: {response_time_avg}')
print(f'Average VRAM usage: {vram_usage_avg}')
print(f'Average RAM usage: {ram_usage_avg}')
print(f'Average total time: {total_time_avg}')

Average response time: 2.32628433945926
Average VRAM usage: 5242.090994371482
Average RAM usage: 102.9576835131332
Average total time: 0.2839072403377111


In [11]:
# save results to txt
with open('results/gemma_ZS_binary2.txt', 'w') as f:
    f.write(f'Accuracy: {accuracy}\n')
    f.write(f'F1 score: {f1}\n')
    f.write(f'Precision: {precision}\n')
    f.write(f'Recall: {recall}\n')
    f.write(f'Average response time: {response_time_avg}\n')
    f.write(f'Average VRAM usage: {vram_usage_avg}\n')
    f.write(f'Average RAM usage: {ram_usage_avg}\n')
    f.write(f'Average total time: {total_time_avg}\n')